# **CLASSIFICATION DE SÉQUENCE POUR UNE BANQUE**

Le but de ce projet est de créer un modèle d'intelligence artificielle se basant sur le modèle **BART**. Ce-dernier est un modèle de séquence-à-séquence extrêmement performant pour la compréhension et la génération de language naturel.

Ainsi, à l'aide du **fine-tuning**, nous allons créer un modèle de **classification d'émails** dans le cas d'une banque en recevrait des milliers par jour afin de pré-traiter ces-derniers et les envoyer dans les départements de l'entreprise correspondant à leurs objets.

*Par exemple un email d'une cliente ne pouvant plus se connecter à l'espace numérique de la banque sera directement envoyé dans le département du support technique.*

In [1]:
# installation des librairies nécessaires

!pip install keras
!pip install torch
!pip install scikit-learn
!pip install transformers

Pour réaliser ce projet, nous aurons besoins de ```scikit-learn``` principalement pour évaluer les performances de notre modèle, ```transformers``` afin d'importer depuis Hugging Face le modèle BART ainsi que le moteur PyTorch : ```torch```. En effet, depuis la version 3.0.0 de Keras, beaucoup de conflits existent entre les librairies ```keras``` et ```transformers```. Nous priviligerons ```PyTorch``` en back-end durant l'utilisation de ```Keras```.

In [13]:
# import des librairies

import sys
import torch
import sklearn
import numpy as np
import pandas as pd
import transformers
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from transformers import Trainer, TrainingArguments
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("librairies importées")

librairies importées


Le but de cet exercice étant de classifier des mails reçu par une banque selon leurs sujets, nous créons un jeu de données simple contenant quelques cas qui sont classés en fonction du département qui traiterons ces mails.

In [3]:
# création du jeu de données fictif

donnees = {
    "email": [
        "Au secours, j'ai perdu ma carte bleue ce matin !",
        "Mon mot de passe sur l'application ne fonctionne plus.",
        "Je voudrais ouvrir un compte joint avec ma femme.",
        "On m'a volé ma carte !",
        "L'application crash quand je clique sur virement."
    ],
    "departement": [
        "Fraude_Urgence",
        "Support_Technique",
        "Conseil_Commercial",
        "Fraude_Urgence",
        "Support_Technique"
    ]
}

df = pd.DataFrame(donnees) # on transforme le dictionnaire python en DataFrame pandas
df.head() # affichage des premières lignes

,email,departement
0,"Au secours, j'ai perdu ma carte bleue ce matin !",Fraude_Urgence
1,Mon mot de passe sur l'application ne fonction...,Support_Technique
2,Je voudrais ouvrir un compte joint avec ma femme.,Conseil_Commercial
3,On m'a volé ma carte !,Fraude_Urgence
4,L'application crash quand je clique sur virement.,Support_Technique


Nous importons un **encodeur** (à l'aide de la librairie `scikit-learn`) afin d'attribuer une valeur à chaque départements différents dans notre banque.

In [4]:
# encodage des données
encodeur = LabelEncoder()
labels_numeriques = encodeur.fit_transform(df["departement"])

Nous devons transposer en **tokens** les différents mots de la séquence afin qu'ils correspondent déjà aux embeddings créés par le modèle "BART".
Nous remarquons que l'import du "tokenizeur" se fait en fonction du modèle.

In [5]:
nom_du_modele = "camembert-base"
tokenizer = AutoTokenizer.from_pretrained(nom_du_modele) #

# créations des tokens
emails_tokenises = tokenizer(
    list(df["email"]),
    padding=True,       # nous remplissons la séquence si celle-ci n'est pas assez longue
    truncation=True,    # à l'inverse, nous tronquons la séquence lorsque celle-ci dépasse les 50 tokens
    max_length=50,      # la séquence gardée finale seras donc de 50 tokens
    return_tensors="pt" # nous indiquons que nous voulons des tensors PyTorch
)

# exemple : première séquence tokenisée
print(emails_tokenises["input_ids"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/811k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.40M [00:00<?, ?B/s]

tensor([   5,  277, 5213,    7,   76,   11,   73, 1576,  155,  637, 5251,   44,
         823,   83,    6])


In [14]:
# nous gardons le nombre de départements uniques
nombre_de_classes = len(df["departement"].unique()) # Ici nous avons 3 départements

# nous téléchargons le modèle et nous indiquons combien de classes il doit prédire à la fin
modele = AutoModelForSequenceClassification.from_pretrained(
    nom_du_modele,
    num_labels=nombre_de_classes
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CamembertForSequenceClassification LOAD REPORT from: camembert-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Afin de pouvoir entraîner à l'aide de `PyTorch`, ce-dernier a besoin d'une petite classe assignant les labels_numériques (correspondant aux départements de l'établissement) à chaque emails tokenisés.

In [7]:
# torch demande de créer une classe afin de bien lire nos emails un par un avec leur étiquette
class banqueDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx): # on attrape l'email numéro "idx"
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["label"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
    def __len__(self):
        return (len(self.labels))

trainSetTorch = banqueDataset(emails_tokenises, labels_numeriques)

La fonction `reglages` correspond aux arguments passés en entrées durant l'entraînement du modèle impliquant le nombre d'epoches (`num_train_epochs`), la taille du lot que le modèle va lire avant de corriger ses poids (`batch_size`), la vitesse à laquelle le modèle "apprend" (`learning_rate`) etc.

In [8]:
reglages = TrainingArguments(
    output_dir="./classificationSequenceBanque",
    num_train_epochs=10,
    per_device_train_batch_size=2, # correspond au batch_size
    learning_rate=5e-5,
    logging_steps=1, # affiche l'évolution à chaque étape
    remove_unused_columns=False # sécurité pour éviter des bugs sur le dataset factice
)

entraineur = Trainer( # "Trainer" correspond à "model.compile()" de Keras
    model=modele,
    args=reglages,
    train_dataset=trainSetTorch
)

print("----------------- DÉBUT DE L'ENTRAÎNEMENT -----------------")
entraineur.train()
print("----------------- FIN DE L'ENTRAÎNEMENT -------------------")

----------------- DÉBUT DE L'ENTRAÎNEMENT -----------------


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,1.098973
2,1.179391
3,1.040656
4,1.071150
5,1.029872
6,1.130142
7,1.041824
8,0.972791
9,1.085942
10,1.077125


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

----------------- FIN DE L'ENTRAÎNEMENT -------------------


Afin de récupérer les différentes métriques d'évaluation nous créons une fonction `compute_metrics` que nous passons en paramètre du `Trainer` de `PyTorch`. Cette fonction comporte, la précision, le rappel, le score F1 ainsi que l'accuracy.

In [9]:
# fonction pour l'évaluation du modèle
def compute_metrics(prediction_eval):
    # pred_evaluation contient les prédictions et les vraies réponses
    logits, labels = prediction_eval

    predictions = np.argmax(logits, axis=-1) # argmax prend le département qui le plus grand score

    # nous utilisons sklearn pour les métriques d'évalution
    precision, recall, f1_score, _ = precision_recall_fscore_support(
        y_true=labels,
        y_pred=predictions,
        average="weighted",
        zero_division=0
    )
    accuracy = accuracy_score(y_true=labels, y_pred=predictions)

    return {
        "accuracy" : accuracy,
        "précision" : precision,
        "rappel" : recall,
        "f1 score" : f1_score
    }

Malgré le peu de matière en ce qui concerne les jeu de données, nous créons un autre jeu de données cette fois-ci d'évaluation afin d'en savoir un peu plus sur les performances de notre modèle sur des données inconnues. Après que ce jeu soit créé, nous devons encoder les départements ("Fraude_Urgence", "Support_Technique", "Conseil_Commercial") ainsi que "tokeniser" les mails.

Enfin, nous passerons en entrée de la fonction `banqueDataset()` les mails tokenisés ainsi que les départements encodés pour créer un jeu de que le moteur de `PyTorch` sera en mesure de les traiter.

In [10]:
donnees_test = {
    "email" : [
        "J'ai bloqué ma carte au distribiteur ce matin !",
        "L'application bug quand je met mon mot de passe."
    ],
    "departement" : ["Fraude_Urgence", "Support_Technique"]
}
df_test = pd.DataFrame(donnees_test)

# nous encodons les "départements" et tokenisons les emails
labels_test = encodeur.transform(df_test["departement"])
emails_test_tokenises = tokenizer(
    list(df_test["email"]),
    padding=True,
    truncation=True,
    max_length=50,
    return_tensors="pt"
)
testSetTorch = banqueDataset(emails_test_tokenises, labels_test)

Maintenant que nous avons le jeu de données qui va nous permettre d'évaluer le modèle, nous passons en entrées du `Trainer()` le **modèle**, le **jeu de données d'évaluation** ainsi que la fonction permettant d'obtenir les **métriques d'évaluation**.

In [11]:
# ÉVALUATION SUR LE JEU DE TEST ---------------------------
entraineur = Trainer(
    model=modele,
    eval_dataset=testSetTorch,
    compute_metrics=compute_metrics # le "Trainer()" s'occupe de passer en paramètres "prediction_eval" dans "compute_metrics"
)
resultats = entraineur.evaluate()

for cle, valeurs in resultats.items():
    print(f"{cle} = {valeurs}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


eval_loss = 0.8083303570747375
eval_model_preparation_time = 0.0103
eval_accuracy = 1.0
eval_précision = 1.0
eval_rappel = 1.0
eval_f1 score = 1.0
eval_runtime = 0.348
eval_samples_per_second = 5.747
eval_steps_per_second = 2.873


Il est normal qu'avec un jeu de test comportant deux cas nous trouvons des scores d'accuracy, de rappel etc. très haut et égal à 1.


En outre, nous pouvons créer une fonction `predire_departement()` prenant en entrée un mail et qui prédit en sortie le département dans lequel le mail est supposé être traité.

In [12]:
# ----------------------- INFÉRENCE --------------------------------------
def predire_departement(texte_email):
    print(f"l'email récupéré est :`{texte_email}`")

    # tokenisation
    inputs = tokenizer(
        list([texte_email]),
        padding=True,
        truncation=True,
        max_length=50,
        return_tensors="pt"
    )

    # prédiction
    # nous interdisons à torch de calculer les gradients car ce n'est pas un entraînement
    with torch.no_grad():
        outputs = modele(**inputs)

    # sortir les prédictions
    scores = outputs.logits # logits représente les valeurs après la dernière couche "dense"
    index_gagnant = torch.argmax(scores, axis=-1).item() # recherche de la plus grande valeur (donc proba)

    label_predit = encodeur.inverse_transform([index_gagnant])[0]

    print(f"le modèle prédit le déppartement : {label_predit}")
    return label_predit

emails = ["Je n'arrive pas à me connecter à mon compte bancaire",
          "J'aimerais ouvrir un compte courant pour mon fils",
          "Mon père s'est fait voler sa carte ce matin"]

prediction = []
for email in emails:
    prediction.append(predire_departement(email))

resultat_inference = pd.DataFrame(emails, prediction)
print(resultat_inference)

l'email récupéré est :`Je n'arrive pas à me connecter à mon compte bancaire`
le modèle prédit le déppartement : Support_Technique
l'email récupéré est :`J'aimerais ouvrir un compte courant pour mon fils`
le modèle prédit le déppartement : Support_Technique
l'email récupéré est :`Mon père s'est fait voler sa carte ce matin`
le modèle prédit le déppartement : Fraude_Urgence
                                                                   0
Support_Technique  Je n'arrive pas à me connecter à mon compte ba...
Support_Technique  J'aimerais ouvrir un compte courant pour mon fils
Fraude_Urgence           Mon père s'est fait voler sa carte ce matin


Ainsi, nous voyons par `resultat_inference` que le modèle a parfaitement compris et intégré les différents départements dans lesquels traiter les mails.